In [1]:
pip install pandas wbgapi

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import wbgapi as wb
from pathlib import Path

In [20]:
# =====================================================
# Config
# =====================================================

START_YEAR = 2010
END_YEAR = 2025

INDICATORS = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "real_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "deposit_interest_rate": "FR.INR.DPST",
    "inflation_rate": "FP.CPI.TOTL.ZG",
    "compulsory_education_years": "SE.COM.DURS",
    "population_65_plus_pct": "SP.POP.65UP.TO.ZS",
    "fertility_rate": "SP.DYN.TFRT.IN",
    "unemployment_rate": "SL.UEM.TOTL.ZS",
    "government_expenditure_pct_gdp": "GC.XPN.TOTL.GD.ZS",
    "government_revenue_pct_gdp": "GC.REV.XGRT.GD.ZS",
    "gdp_deflator_inflation": "NY.GDP.DEFL.KD.ZG"
}


In [21]:
countries = pd.DataFrame(list(wb.economy.list()))

# Remove agregados (World, OECD, etc.)
countries = countries[countries["aggregate"] == False]

country_codes = countries["id"].tolist()

print(f"{len(country_codes)} países encontrados.")

217 países encontrados.


In [22]:
df = wb.data.DataFrame(
    "NY.GDP.PCAP.CD",
    economy=country_codes,
    time=range(2010, 2025),
    labels=True
)

# transforma índice em coluna
df = df.reset_index()

# formato longo
df = df.melt(
    id_vars=["economy", "Country"],
    var_name="year",
    value_name="gdp_per_capita"
)

# limpa o ano
df["year"] = (
    df["year"]
    .str.replace("YR", "", regex=False)
    .astype(int)
)

print(df.head())

  economy       Country  year  gdp_per_capita
0     ZWE      Zimbabwe  2010      902.010759
1     ZMB        Zambia  2010     1451.106160
2     ZAF  South Africa  2010     7973.471958
3     YEM   Yemen, Rep.  2010     1155.203053
4     XKX        Kosovo  2010     2987.545196


In [23]:
# PAÍSES
# =====================================================

countries = pd.DataFrame(list(wb.economy.list()))
countries = countries[countries["aggregate"] == False]

country_codes = countries["id"].tolist()

print(f"{len(country_codes)} países encontrados")

# INDICADORES
# =====================================================

INDICATORS = {
    "gdp_per_capita": "NY.GDP.PCAP.CD",
    "real_gdp_growth": "NY.GDP.MKTP.KD.ZG",
    "inflation_rate": "FP.CPI.TOTL.ZG",
    "compulsory_education_years": "SE.COM.DURS",
    "population_65_plus_pct": "SP.POP.65UP.TO.ZS",
    "fertility_rate": "SP.DYN.TFRT.IN",
    "unemployment_rate": "SL.UEM.TOTL.ZS",
    "government_expenditure_pct_gdp": "GC.XPN.TOTL.GD.ZS",
    "government_revenue_pct_gdp": "GC.REV.XGRT.GD.ZS",
    "gdp_deflator_inflation": "NY.GDP.DEFL.KD.ZG",
    "current_account_pct_gdp": "BN.CAB.XOKA.GD.ZS",
    "age_dependency_ratio": "SP.POP.DPND",
    "gross_capital_formation_pct_gdp": "NE.GDI.TOTL.ZS",
    "total_reserves_months_imports": "FI.RES.TOTL.MO",
    "savings_pct_gdp": "NY.GNS.ICTR.ZS",
    "exports_pct_gdp": "NE.EXP.GNFS.ZS"
}

# FUNÇÃO DE DOWNLOAD
# =====================================================

def download_indicator(indicator_code, variable_name):

    print(f"Baixando {variable_name}...")

    df = wb.data.DataFrame(
        indicator_code,
        economy=country_codes,
        time=range(2010, 2024),
        labels=True
    )

    df = df.reset_index()

    df = df.melt(
        id_vars=["economy", "Country"],
        var_name="year",
        value_name=variable_name
    )

    df["year"] = (
        df["year"]
        .str.replace("YR", "", regex=False)
        .astype(int)
    )

    return df


# PRIMEIRO INDICADOR
# =====================================================

first_var = list(INDICATORS.keys())[0]
first_code = INDICATORS[first_var]

panel = download_indicator(first_code, first_var)

# MERGE DOS DEMAIS
# =====================================================

for var, code in list(INDICATORS.items())[1:]:

    temp = download_indicator(code, var)

    panel = panel.merge(
        temp[
            ["economy", "year", var]
        ],
        on=["economy", "year"],
        how="left"
    )

# RENOMEIA
# =====================================================

panel = panel.rename(columns={
    "economy": "country_code",
    "Country": "country"
})

# ORDENA
# =====================================================

panel = panel.sort_values(
    ["country", "year"]
).reset_index(drop=True)

# RESULTADO
# =====================================================

print(panel.head())

print("\nShape:")
print(panel.shape)

print("\nMissing Values:")
print(
    panel.isna()
    .mean()
    .sort_values(ascending=False)
)

# SALVA
# =====================================================

panel.to_csv(
    "wdi_fiscal_risk_panel.csv",
    index=False
)

print("\nArquivo salvo!")

217 países encontrados
Baixando gdp_per_capita...
Baixando real_gdp_growth...
Baixando inflation_rate...
Baixando compulsory_education_years...
Baixando population_65_plus_pct...
Baixando fertility_rate...
Baixando unemployment_rate...
Baixando government_expenditure_pct_gdp...
Baixando government_revenue_pct_gdp...
Baixando gdp_deflator_inflation...
Baixando current_account_pct_gdp...
Baixando age_dependency_ratio...
Baixando gross_capital_formation_pct_gdp...
Baixando total_reserves_months_imports...
Baixando savings_pct_gdp...
Baixando exports_pct_gdp...
  country_code      country  year  gdp_per_capita  real_gdp_growth  \
0          AFG  Afghanistan  2010      560.621505        14.362441   
1          AFG  Afghanistan  2011      606.694676         0.426355   
2          AFG  Afghanistan  2012      651.417134        12.752287   
3          AFG  Afghanistan  2013      637.087099         5.600745   
4          AFG  Afghanistan  2014      625.054942         2.724543   

   inflation_ra

In [28]:
coverage = (
    panel
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                        0.00
country                             0.00
year                                0.00
population_65_plus_pct              0.00
age_dependency_ratio                0.00
fertility_rate                      0.00
gdp_per_capita                      2.83
real_gdp_growth                     3.49
gdp_deflator_inflation              3.72
compulsory_education_years         10.76
unemployment_rate                  13.96
inflation_rate                     14.78
current_account_pct_gdp            15.01
exports_pct_gdp                    16.62
gross_capital_formation_pct_gdp    20.08
total_reserves_months_imports      23.73
savings_pct_gdp                    26.40
government_revenue_pct_gdp         38.51
government_expenditure_pct_gdp     39.47
dtype: float64


In [25]:
missing_by_country = (
    panel
    .isna()
    .groupby(panel["country"])
    .mean()
    .mul(100)
    .round(2)
)

print(missing_by_country.head())

                country_code  country  year  gdp_per_capita  real_gdp_growth  \
country                                                                        
Afghanistan              0.0      0.0   0.0            0.00             0.00   
Albania                  0.0      0.0   0.0            0.00             0.00   
Algeria                  0.0      0.0   0.0            0.00             0.00   
American Samoa           0.0      0.0   0.0            7.14             7.14   
Andorra                  0.0      0.0   0.0            0.00             0.00   

                inflation_rate  compulsory_education_years  \
country                                                      
Afghanistan                0.0                         0.0   
Albania                    0.0                         0.0   
Algeria                    0.0                         0.0   
American Samoa           100.0                       100.0   
Andorra                  100.0                         0.0   

    

In [26]:
# Colunas que não são variáveis
id_cols = ["country", "country_code", "year"]

vars_cols = [c for c in panel.columns if c not in id_cols]

# Percentual de missing por país e variável
missing_country_var = (
    panel
    .groupby("country_code")[vars_cols]
    .apply(lambda x: x.isna().mean())
)

# Países que possuem alguma variável 100% nula
countries_to_drop = (
    missing_country_var
    .eq(1)
    .any(axis=1)
)

countries_to_drop = countries_to_drop[
    countries_to_drop
].index.tolist()

print(f"Países removidos: {len(countries_to_drop)}")
print(countries_to_drop)

Países removidos: 99
['ABW', 'AND', 'ASM', 'ATG', 'BDI', 'BEN', 'BFA', 'BMU', 'BOL', 'BRB', 'BRN', 'BTN', 'BWA', 'CAF', 'CHI', 'CHN', 'CIV', 'COM', 'CUB', 'CUW', 'CYM', 'DJI', 'DMA', 'DZA', 'ERI', 'FJI', 'FRO', 'FSM', 'GIB', 'GIN', 'GMB', 'GNB', 'GNQ', 'GRD', 'GRL', 'GUM', 'GUY', 'HKG', 'HTI', 'IDN', 'IMN', 'IRN', 'JAM', 'JPN', 'KHM', 'KIR', 'KNA', 'KWT', 'LBR', 'LBY', 'LCA', 'LIE', 'MAF', 'MCO', 'MHL', 'MLI', 'MMR', 'MNE', 'MNP', 'MOZ', 'MRT', 'NCL', 'NER', 'NGA', 'NRU', 'OMN', 'PAK', 'PLW', 'PNG', 'PRI', 'PRK', 'PSE', 'PYF', 'QAT', 'SEN', 'SLB', 'SLE', 'SMR', 'SOM', 'SSD', 'STP', 'SUR', 'SXM', 'SYC', 'SYR', 'TCA', 'TCD', 'TGO', 'TKM', 'TTO', 'TUV', 'VCT', 'VEN', 'VGB', 'VIR', 'VNM', 'VUT', 'XKX', 'YEM']


In [29]:
panel_clean = panel[
    ~panel["country_code"].isin(countries_to_drop)
].copy()

print(panel.shape)
print(panel_clean.shape)

(3038, 19)
(1652, 19)


In [30]:
coverage = (
    panel_clean
    .isna()
    .mean()
    .mul(100)
    .round(2)
    .sort_values()
)

print(coverage)

country_code                        0.00
country                             0.00
year                                0.00
gdp_per_capita                      0.00
real_gdp_growth                     0.00
population_65_plus_pct              0.00
gdp_deflator_inflation              0.00
fertility_rate                      0.00
age_dependency_ratio                0.00
unemployment_rate                   0.18
compulsory_education_years          0.30
inflation_rate                      1.76
current_account_pct_gdp             1.82
exports_pct_gdp                     2.24
gross_capital_formation_pct_gdp     2.30
total_reserves_months_imports       2.78
savings_pct_gdp                     4.18
government_revenue_pct_gdp          9.02
government_expenditure_pct_gdp     10.53
dtype: float64


In [33]:
panel_clean.head()

,country_code,country,year,gdp_per_capita,real_gdp_growth,inflation_rate,compulsory_education_years,population_65_plus_pct,fertility_rate,unemployment_rate,government_expenditure_pct_gdp,government_revenue_pct_gdp,gdp_deflator_inflation,current_account_pct_gdp,age_dependency_ratio,gross_capital_formation_pct_gdp,total_reserves_months_imports,savings_pct_gdp,exports_pct_gdp
0,AFG,Afghanistan,2010,560.621505,14.362441,2.178538,9.0,2.306615,6.195,7.809,50.863005,11.091949,3.814630,-3.643314,103.144456,NaN,10.457387,NaN,NaN
1,AFG,Afghanistan,2011,606.694676,0.426355,11.804186,9.0,2.322140,6.094,7.830,59.484776,11.510161,16.593347,-12.619538,101.458308,NaN,10.343973,NaN,NaN
2,AFG,Afghanistan,2012,651.417134,12.752287,6.441213,9.0,2.337911,5.985,7.875,42.329251,10.200625,7.301756,-25.870681,99.773131,NaN,8.677209,NaN,NaN
3,AFG,Afghanistan,2013,637.087099,5.600745,7.385772,9.0,2.349769,5.879,7.921,42.093684,9.405609,4.822785,-25.290059,98.063559,NaN,8.588138,NaN,NaN
4,AFG,Afghanistan,2014,625.054942,2.724543,4.673996,9.0,2.356536,5.770,7.915,44.589299,8.613731,0.566945,-15.772420,96.346394,NaN,10.702695,NaN,NaN


In [37]:
id_cols = ["country", "country_code", "year"]

vars_cols = [
    c for c in panel_clean.columns
    if c not in id_cols
]

country_medians = (
    panel_clean
    .groupby("country_code")[vars_cols]
    .median()
    .reset_index()
)

print(country_medians.head())

  country_code  gdp_per_capita  real_gdp_growth  inflation_rate  \
0          AFG      523.775993         2.263629        4.824974   
1          AGO     3278.932786         1.683030       15.775305   
2          ALB     4899.978820         2.718336        2.007360   
3          ARE    46536.983276         4.592834        1.359303   
4          ARG    12702.122748         0.689452       50.978841   

   compulsory_education_years  population_65_plus_pct  fertility_rate  \
0                         9.0                2.356017          5.4875   
1                        10.0                2.722321          5.6430   
2                         9.0               13.112802          1.5205   
3                        12.0                1.498832          1.3580   
4                        14.0               11.237970          2.2045   

   unemployment_rate  government_expenditure_pct_gdp  \
0            10.6500                       43.126009   
1            16.5180                       19.

In [38]:
id_cols = ["country", "country_code", "year"]

vars_cols = [
    c for c in panel_clean.columns
    if c not in id_cols
]

panel_imputed = panel_clean.copy()

panel_imputed[vars_cols] = (
    panel_imputed
    .groupby("country_code")[vars_cols]
    .transform(
        lambda x: x.fillna(x.median())
    )
)

In [40]:
panel_imputed.isna().sum().sort_values(ascending=False)

country_code                       0
country                            0
year                               0
gdp_per_capita                     0
real_gdp_growth                    0
inflation_rate                     0
compulsory_education_years         0
population_65_plus_pct             0
fertility_rate                     0
unemployment_rate                  0
government_expenditure_pct_gdp     0
government_revenue_pct_gdp         0
gdp_deflator_inflation             0
current_account_pct_gdp            0
age_dependency_ratio               0
gross_capital_formation_pct_gdp    0
total_reserves_months_imports      0
savings_pct_gdp                    0
exports_pct_gdp                    0
dtype: int64

In [41]:
panel_imputed.to_csv("panel_clean.csv", index=False)